# NLP Word Count Pipeline — Sustainability Reports

Processes PDF annual reports from `data_ar_kam/`, extracts text (PyMuPDF + Gemini OCR for scanned pages), counts dictionary terms from `dt_kam_wordcount.csv`, and outputs structured word count results.

**Features:**
- Hybrid text extraction: direct text + Gemini OCR for scanned/image pages
- Parallel processing with ThreadPoolExecutor
- Checkpoint/resume support (safe to interrupt and re-run)
- Per-page diagnostics and token usage tracking

In [ ]:
!pip install PyMuPDF google-genai pandas tqdm Pillow

In [ ]:
%load_ext autoreload
%autoreload 2

import pandas as pd
from pathlib import Path
import json

from src import config
from src.logger import setup_logger, get_logger
from src.utils import (
    ensure_dirs, discover_pdf_files, parse_filename,
    load_dictionary, load_ledger, save_results,
)
from src.pipeline import run_pipeline

In [ ]:
# Configuration Preview
print("=" * 50)
print("Pipeline Configuration")
print("=" * 50)
print(f"PROJECT_ID:       {config.PROJECT_ID}")
print(f"MODEL_ID:         {config.MODEL_ID}")
print(f"LOCATION:         {config.LOCATION}")
print(f"OCR_MODE:         {config.OCR_MODE}")
print(f"BATCH_SIZE:       {config.BATCH_SIZE}")
print(f"MAX_FILES:        {config.MAX_FILES}")
print(f"MAX_WORKERS:      {config.MAX_WORKERS}")
print(f"OCR_IMAGE_DPI:    {config.OCR_IMAGE_DPI}")
print(f"MIN_TEXT_THRESH:   {config.MIN_TEXT_THRESHOLD}")
print()

# Verify paths
errors = []
if not config.PDF_DIR.exists():
    errors.append(f"PDF_DIR not found: {config.PDF_DIR}")
if not config.DICTIONARY_PATH.exists():
    errors.append(f"DICTIONARY_PATH not found: {config.DICTIONARY_PATH}")
if not config.SERVICE_ACCOUNT_PATH.exists():
    errors.append(f"SERVICE_ACCOUNT_PATH not found: {config.SERVICE_ACCOUNT_PATH}")

if errors:
    for e in errors:
        print(f"ERROR: {e}")
    raise FileNotFoundError("Missing required files. See errors above.")
else:
    print("All paths verified OK")
    print(f"  PDF_DIR:              {config.PDF_DIR}")
    print(f"  DICTIONARY_PATH:      {config.DICTIONARY_PATH}")
    print(f"  SERVICE_ACCOUNT_PATH: {config.SERVICE_ACCOUNT_PATH}")

In [ ]:
# Load Dictionary
dictionary_df = load_dictionary()
print(f"Shape: {dictionary_df.shape}")
print(f"Unique dimensions: {dictionary_df['Dimensions'].nunique()}")
print()
print("Sample rows:")
dictionary_df.head(10)

In [ ]:
# Discover PDFs
setup_logger()
pdf_files = discover_pdf_files(max_files=config.MAX_FILES)
print(f"Total PDFs found: {len(pdf_files)}")
print()
print("First 10 filenames:")
for f in pdf_files[:10]:
    print(f"  {f.name}")

print()
print("Sample filename parsing:")
for f in pdf_files[:3]:
    try:
        code, year = parse_filename(f)
        print(f"  {f.name} -> emiten_code={code}, year={year}")
    except ValueError as e:
        print(f"  {f.name} -> ERROR: {e}")

## Run Pipeline

This cell runs the full pipeline. It processes `MAX_FILES` PDFs in batches of `BATCH_SIZE`.

**Resume support:** Safe to interrupt and re-run — the pipeline skips already-processed files automatically.

In [ ]:
wordcount_df, summary_df = run_pipeline()

## Results

In [ ]:
# Word Count Results
if wordcount_df is None or wordcount_df.empty:
    wc_path = config.OUTPUT_DIR / "wordcount_results.csv"
    if wc_path.exists():
        wordcount_df = pd.read_csv(wc_path)

print(f"Shape: {wordcount_df.shape}")
print(f"Unique Emiten Codes: {wordcount_df['Emiten Code'].nunique()}")
print(f"Unique Dimensions: {wordcount_df['Dimensions'].nunique()}")
print()
print("Head (20 rows):")
display(wordcount_df.head(20))
print()
print("Tail (10 rows):")
display(wordcount_df.tail(10))

In [ ]:
# Process Summary
if summary_df is None or summary_df.empty:
    sum_path = config.OUTPUT_DIR / "process_summary.csv"
    if sum_path.exists():
        summary_df = pd.read_csv(sum_path)

print(f"Shape: {summary_df.shape}")
print()
print("Status distribution:")
print(summary_df["status"].value_counts())
print()
display(summary_df.head(10))

# Show failed files if any
failed = summary_df[summary_df["status"] == "failed"]
if not failed.empty:
    print(f"\nFailed files ({len(failed)}):")
    for _, row in failed.iterrows():
        print(f"  {row['file_name']}: {row['error_message']}")
else:
    print("\nNo failed files!")

## Token Usage & Cost Analysis

In [ ]:
# Token Usage Analysis
token_path = config.TOKEN_USAGE_PATH
if token_path.exists():
    token_df = pd.read_csv(token_path)

    total_input = token_df["prompt_tokens"].sum()
    total_output = token_df["output_tokens"].sum()
    total_input_cost = total_input / 1_000_000 * config.PRICE_INPUT_PER_M
    total_output_cost = total_output / 1_000_000 * config.PRICE_OUTPUT_PER_M
    total_cost = total_input_cost + total_output_cost

    num_pdfs = summary_df[summary_df["status"] == "success"].shape[0] if not summary_df.empty else 1
    avg_cost_per_pdf = total_cost / num_pdfs if num_pdfs > 0 else 0
    projected_full_cost = avg_cost_per_pdf * 2323

    print("=" * 50)
    print("Token Usage & Cost Summary")
    print("=" * 50)
    print(f"Total OCR API calls:    {len(token_df)}")
    print(f"Total input tokens:     {total_input:,}")
    print(f"Total output tokens:    {total_output:,}")
    print(f"Input cost:             ${total_input_cost:.6f}")
    print(f"Output cost:            ${total_output_cost:.6f}")
    print(f"Total cost (this run):  ${total_cost:.6f}")
    print(f"Avg cost per PDF:       ${avg_cost_per_pdf:.6f}")
    print(f"Projected cost (2,323 PDFs): ${projected_full_cost:.4f}")
    print("=" * 50)
else:
    print("No token usage data found. Pipeline may not have used OCR.")

## Page-Level Diagnostics

In [ ]:
# Page Diagnostics
diag_path = config.PAGE_DIAGNOSTICS_PATH
if diag_path.exists():
    diag_df = pd.read_csv(diag_path)

    print(f"Total pages processed: {len(diag_df)}")
    print()

    print("Classification distribution:")
    print(diag_df["classification"].value_counts())
    print()

    print("Extraction method distribution:")
    print(diag_df["extraction_method"].value_counts())
    print()

    print("Top 10 PDFs by OCR page count:")
    ocr_counts = diag_df[diag_df["classification"] == "image"].groupby("file_name").size().nlargest(10)
    print(ocr_counts)
    print()

    print("Average text length by extraction method:")
    print(diag_df.groupby("extraction_method")["final_text_length"].mean())
    print()

    zero_text = diag_df[diag_df["final_text_length"] == 0]
    print(f"Pages with zero extracted text: {len(zero_text)}")
    if not zero_text.empty:
        display(zero_text[["file_name", "page_number", "classification", "error"]].head(20))
else:
    print("No page diagnostics data found.")

## Validation & Sanity Checks

In [ ]:
# Validation & Sanity Checks
warnings = []

# 1. PDF count
all_pdfs = list(config.PDF_DIR.glob("*.pdf"))
processed_count = len(summary_df) if not summary_df.empty else 0
print(f"1. PDFs found: {len(all_pdfs)}, Processed: {processed_count}")

# 2. Dictionary entries
dict_count = len(dictionary_df)
print(f"2. Dictionary entries: {dict_count} (expected ~102)")
if dict_count != 102:
    warnings.append(f"Dictionary has {dict_count} entries, expected 102")

# 3. Duplicate check
if not wordcount_df.empty:
    dup_cols = ["Emiten Code", "Year", "Dimensions", "Wordlist"]
    dups = wordcount_df.duplicated(subset=dup_cols, keep=False)
    dup_count = dups.sum()
    print(f"3. Duplicate (code, year, dim, wordlist) rows: {dup_count}")
    if dup_count > 0:
        warnings.append(f"{dup_count} duplicate rows found (may be expected if dictionary has duplicates)")

# 4. Missing values
if not wordcount_df.empty:
    na_counts = wordcount_df[["Emiten Code", "Year", "Dimensions", "Wordlist", "Word count"]].isna().sum()
    total_na = na_counts.sum()
    print(f"4. Missing values in key columns: {total_na}")
    if total_na > 0:
        print(na_counts[na_counts > 0])
        warnings.append(f"{total_na} missing values found")

# 5. Zero-count analysis
if not wordcount_df.empty:
    term_totals = wordcount_df.groupby("Wordlist")["Word count"].sum()
    zero_terms = term_totals[term_totals == 0]
    print(f"5. Wordlist terms with 0 matches across ALL files: {len(zero_terms)}/{len(term_totals)}")
    if len(zero_terms) > 0:
        print(f"   Zero-match terms: {list(zero_terms.index[:20])}")

# 6. OCR ratio
if diag_path.exists():
    total_pages = len(diag_df)
    ocr_pages = len(diag_df[diag_df["extraction_method"] == "gemini_ocr"])
    ocr_ratio = ocr_pages / total_pages * 100 if total_pages > 0 else 0
    print(f"6. OCR ratio: {ocr_pages}/{total_pages} pages ({ocr_ratio:.1f}%)")

print()
if warnings:
    print(f"Warnings ({len(warnings)}):")
    for w in warnings:
        print(f"  - {w}")
else:
    print("All checks passed!")

## Export Extracted Text to .txt Files

Save the extracted text per PDF as `.txt` files in `output/extracted_text/` for manual inspection.
This helps verify what text was actually extracted (both PyMuPDF and OCR).

In [ ]:
# Export extracted text from already-processed PDFs (PyMuPDF only, no OCR cost)
# This is a quick re-extraction to create .txt files for inspection
from src.text_export import batch_export_texts, export_text_from_pdf

# Export text for all PDFs that were processed in batch 1 (first 500)
# Uses PyMuPDF only — fast, no API cost
exported_count = batch_export_texts(max_files=500, skip_existing=True)
print(f"Exported {exported_count} text files to output/extracted_text/")
print()

# Show a sample extracted text file
import os
txt_dir = config.EXTRACTED_TEXT_DIR
txt_files = sorted(txt_dir.glob("*.txt"))[:3]
for tf in txt_files:
    size = os.path.getsize(tf)
    print(f"  {tf.name} ({size:,} bytes)")
    # Show first 500 chars
    content = tf.read_text(encoding="utf-8")[:500]
    print(f"  Preview: {content[:200]}...")
    print()

In [ ]:
# Full-text export WITH OCR (uses Gemini API — costs tokens but captures scanned pages)
# Uncomment and run if you want OCR text saved too (only for files not yet exported)

# from src.pipeline import init_gemini_client
# client = init_gemini_client()
# from src.text_export import batch_export_with_ocr
# exported_ocr = batch_export_with_ocr(max_files=500, skip_existing=True, client=client)
# print(f"Exported {exported_ocr} text files (with OCR) to output/extracted_text/")

## Continue Processing — Batch 2 (Remaining Files)

After batch 1 (first 500 files), use this section to process the remaining PDFs.
The pipeline automatically skips already-processed files via the ledger.

**How it works:**
- Set `MAX_FILES` to the new total (e.g. 1000 for next 500, or `None` for ALL 2,323)
- The pipeline loads the ledger, skips files already marked as `success`, and processes only the remaining ones
- Results are appended to the existing output files

In [ ]:
# ============================================================
# BATCH 2 CONFIGURATION — Edit these values before running
# ============================================================

# Option A: Process next 500 files (files 501-1000)
NEXT_MAX_FILES = 1000

# Option B: Process ALL remaining files (uncomment below)
# NEXT_MAX_FILES = None  # None = process all 2,323 PDFs

# Option C: Custom range (e.g. next 200 files)
# NEXT_MAX_FILES = 700  # 500 already done + 200 new = 700 total

NEXT_BATCH_SIZE = 50  # files per batch (same as batch 1)

print(f"Batch 2 config:")
print(f"  MAX_FILES:  {NEXT_MAX_FILES} {'(all files)' if NEXT_MAX_FILES is None else ''}")
print(f"  BATCH_SIZE: {NEXT_BATCH_SIZE}")
print()

# Check current progress
ledger = load_ledger()
done = sum(1 for v in ledger.values() if v.get("status") == "success")
total_pdfs = len(list(config.PDF_DIR.glob("*.pdf")))
remaining = total_pdfs - done
print(f"Progress so far: {done}/{total_pdfs} PDFs processed ({remaining} remaining)")

In [ ]:
# Run Batch 2
# This will skip already-processed files and only process new ones
wordcount_df, summary_df = run_pipeline(max_files=NEXT_MAX_FILES, batch_size=NEXT_BATCH_SIZE)

In [ ]:
# Batch 2 Results Summary
print("=" * 60)
print("Batch 2 Results")
print("=" * 60)

# Reload latest results
wc_df = pd.read_csv(config.OUTPUT_DIR / "wordcount_results.csv")
sum_df = pd.read_csv(config.OUTPUT_DIR / "process_summary.csv")

print(f"Total word count rows: {len(wc_df)}")
print(f"Total files processed: {len(sum_df)}")
print(f"Status distribution:")
print(sum_df["status"].value_counts())
print()

# Non-zero word counts
nonzero = wc_df[wc_df["Word count"] > 0]
print(f"Non-zero word counts: {len(nonzero)} / {len(wc_df)} ({len(nonzero)/len(wc_df)*100:.1f}%)")
print()

# Top matched terms
if not nonzero.empty:
    top_terms = nonzero.groupby("Wordlist")["Word count"].sum().nlargest(20)
    print("Top 20 matched wordlist terms:")
    print(top_terms)
    print()

# Updated progress
ledger = load_ledger()
done = sum(1 for v in ledger.values() if v.get("status") == "success")
total_pdfs = len(list(config.PDF_DIR.glob("*.pdf")))
print(f"Overall progress: {done}/{total_pdfs} PDFs ({total_pdfs - done} remaining)")